In [1]:
import requests

url = "https://api.openaq.org/v3/locations?countries_id=65&limit=1000"
headers = {"X-API-Key": "a4f714253a2b3b60ff024968b80f3a0581086849f2c523197bffe2043e735d80"}

response = requests.get(url, headers=headers)
data = response.json()

for loc in data["results"]:
    print(
        f"ID: {loc.get('id')} | "
        f"Name: {loc.get('name')} | "
        f"Locality: {loc.get('locality')}"
    )

ID: 2536 | Name: US Diplomatic Post: Pristina | Locality: None
ID: 7674 | Name: US Diplomatic Post: Pristina | Locality: Pristina
ID: 7928 | Name: Peje | Locality: Peje
ID: 7929 | Name: Mitrovice | Locality: Mitrovice
ID: 7930 | Name: Prizren | Locality: Prizren
ID: 7931 | Name: AQMS Prishtine - IHMK | Locality: AQMS Prishtine - IHMK
ID: 7932 | Name: Gjilan | Locality: Gjilan
ID: 7933 | Name: AQMS Prishtine Rilindje | Locality: AQMS Prishtine Rilindje
ID: 7934 | Name: Drenas-Municipality | Locality: Drenas-Municipality
ID: 7935 | Name: AQMS Hani i Elezit | Locality: Podgorica
ID: 9337 | Name: Pristina | Locality: N/A
ID: 494935 | Name: AQMS Obiliq | Locality: AQMS Obiliq
ID: 494936 | Name: Mobile | Locality: Mobile
ID: 494937 | Name: Brezovica | Locality: Brezovica
ID: 494938 | Name: Palaj | Locality: Palaj
ID: 494939 | Name: AQMS Dardhishte | Locality: AQMS Dardhishte
ID: 494941 | Name: AQMS Peje | Locality: AQMS Peje
ID: 921111 | Name: Drenas | Locality: Drenas
ID: 921120 | Name: Gji

In [2]:
import os
import glob
import pandas as pd

base_folder = r"C:\Users\Festina\Desktop\openaq_prishtina"
locations = [2536, 7674, 7931, 7933, 9337]

for loc in locations:
    print(f"\n========== LOCATION {loc} ==========")
    
    folder = os.path.join(base_folder, str(loc))
    files = glob.glob(os.path.join(folder, "**", "*.csv.gz"), recursive=True)
    
    dfs = []
    
    for file in files:
        try:
            df = pd.read_csv(file, compression="gzip")
            dfs.append(df)
        except:
            pass
    
    if not dfs:
        print("Nuk ka te dhena.")
        continue
    
    df_loc = pd.concat(dfs, ignore_index=True)
    
    if "parameter" in df_loc.columns:
        params = df_loc["parameter"].dropna().unique()
        print(f"Nr i parametrave: {len(params)}")
        print("Parametrat:", params)
    else:
        print("Kolona 'parameter' nuk ekziston")
    
    if "datetime" in df_loc.columns:
        df_loc["datetime"] = pd.to_datetime(df_loc["datetime"], errors="coerce")
        print("Data min:", df_loc["datetime"].min())
        print("Data max:", df_loc["datetime"].max())
    else:
        print("Kolona 'datetime' nuk ekziston")


========== LOCATION 2536 ==========
Nuk ka te dhena.

========== LOCATION 7674 ==========
Nuk ka te dhena.

========== LOCATION 7931 ==========
Nuk ka te dhena.

========== LOCATION 7933 ==========
Nuk ka te dhena.

========== LOCATION 9337 ==========
Nuk ka te dhena.


In [4]:
# Assuming os, glob, pd are already imported from previous cells
base_folder = r"C:\Users\Festina\Desktop\openaq_prishtina"
locations = [7931, 7933]

all_dfs = []

for loc in locations:
    folder = os.path.join(base_folder, str(loc))
    files = glob.glob(os.path.join(folder, "**", "*.csv.gz"), recursive=True)

    for file in files:
        filename = os.path.basename(file)

        if any(year in filename for year in ["2023", "2024", "2025", "2026"]):
            try:
                df = pd.read_csv(file, compression="gzip")
                df["location_id"] = loc
                all_dfs.append(df)
                print(f"Loaded: {filename}")
            except Exception as e:
                print(f"Error: {filename}")

if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
else:
    final_df = pd.DataFrame()  # Empty DataFrame if no data loaded

print(final_df.shape)

print(final_df.shape)

(0, 0)
(0, 0)


In [7]:
if final_df.empty or 'location_id' not in final_df.columns:
    print("No data loaded or 'location_id' column missing in final_df.")
    df_avg = pd.DataFrame()  # Empty DataFrame to avoid errors
else:
    df = final_df[final_df["location_id"].isin([7931, 7933])].copy()

    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

    df_avg = (
        df.groupby(["datetime", "parameter"])["value"]
        .mean()
        .reset_index()
    )

print(df_avg.head())

No data loaded or 'location_id' column missing in final_df.
Empty DataFrame
Columns: []
Index: []


In [9]:
if df_avg.empty or 'datetime' not in df_avg.columns or 'parameter' not in df_avg.columns or 'value' not in df_avg.columns:
    print("df_avg is empty or missing required columns ('datetime', 'parameter', 'value'). Cannot create pivot table.")
else:
    df_pivot = df_avg.pivot(index="datetime", columns="parameter", values="value").reset_index()
    df_pivot = df_pivot.sort_values("datetime")
    output_path = r"C:\Users\Festina\Desktop\openaq_prishtina\prishtina_air_quality_2023_2025.csv"
    df_pivot.to_csv(output_path, index=False)
    print(df_pivot.head())
    print(df_pivot.shape)
    print(output_path)

df_avg is empty or missing required columns ('datetime', 'parameter', 'value'). Cannot create pivot table.
